# OncologyBreastScanner — v2 (surgical install)

**Keep cells 1-3 from v1 unchanged.** They are already run:
1. `drive.mount()`  
2. `!pip install`  
3. `kaggle_path = kagglehub.dataset_download(...)`

**Replace everything below with the cells in this notebook.**

Changes from v1: filters out full mammograms (mean<30 = mostly black after 224×224 resize). Only ROI crops are used.

## Cell 4 — Prepare Dataset (with dark image filter)

In [ ]:
import pandas as pd, os, cv2, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

KAGGLE_DIR = Path(kaggle_path)
DATA_DIR = Path('/content/data2')
IMAGES_DIR = DATA_DIR / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_LABELS = [
    'Density1Benign', 'Density1Malignant',
    'Density2Benign', 'Density2Malignant',
    'Density3Benign', 'Density3Malignant',
    'Density4Benign', 'Density4Malignant',
]

def pathology_to_binary(p):
    p = str(p).strip().upper()
    return 'Benign' if 'BENIGN' in p else 'Malignant' if 'MALIGNANT' in p else None

def get_density(row):
    for col in ['breast density', 'breast_density', 'density']:
        val = row.get(col)
        if val is not None and not pd.isna(val):
            return val
    return None

# --- Scan CSVs, map to JPEGs, filter dark ---
csv_dir = KAGGLE_DIR / 'csv'
jpeg_dir = KAGGLE_DIR / 'jpeg'

records = []
skipped = 0

for csv_name in [
    'mass_case_description_train_set.csv',
    'mass_case_description_test_set.csv',
    'calc_case_description_train_set.csv',
    'calc_case_description_test_set.csv',
]:
    csv_path = csv_dir / csv_name
    if not csv_path.exists():
        continue
    df = pd.read_csv(csv_path)
    for _, row in tqdm(df.iterrows(), total=len(df), desc=csv_name):
        density = get_density(row)
        pathology = pathology_to_binary(row.get('pathology'))
        try:
            d = int(float(str(density).strip()))
        except Exception:
            continue
        if d is None or pathology is None or d < 1 or d > 4:
            continue
        label = f'Density{d}{pathology}'
        if label not in CLASS_LABELS:
            continue

        jpeg_path = None
        for col in ['cropped image file path', 'image file path']:
            dcm = row.get(col)
            if not dcm or pd.isna(dcm):
                continue
            parts = str(dcm).strip().split('/')
            if len(parts) < 2:
                continue
            uid = parts[-2]
            uid_dir = jpeg_dir / uid
            if not uid_dir.is_dir():
                continue
            jpgs = sorted(f for f in os.listdir(uid_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
            if jpgs:
                jpeg_path = uid_dir / jpgs[0]
                break

        if not jpeg_path:
            continue

        img = cv2.imread(str(jpeg_path))
        if img is None:
            continue
        small = cv2.resize(img, (224, 224))
        if small.mean() < 30:
            skipped += 1
            continue

        records.append({'source': str(jpeg_path), 'label': label})

print(f'\nKept: {len(records)}   Skipped (dark, likely full mammogram): {skipped}')

# --- Copy to flat directory ---
for i, rec in enumerate(tqdm(records, desc='Copying images')):
    dst = IMAGES_DIR / f"{rec['label']}_{i:05d}.png"
    img = cv2.imread(rec['source'])
    if img is None:
        continue
    img = cv2.resize(img, (224, 224))
    cv2.imwrite(str(dst), img)

# --- Train/Test split ---
all_files = sorted(os.listdir(IMAGES_DIR))
all_labels = [f.rsplit('_', 1)[0] for f in all_files]

train_files, test_files = train_test_split(
    all_files, test_size=0.15, stratify=all_labels, random_state=42
)

train_df = pd.DataFrame({
    'filename': train_files,
    'label': [f.rsplit('_', 1)[0] for f in train_files]
})
test_df = pd.DataFrame({
    'filename': test_files,
    'label': [f.rsplit('_', 1)[0] for f in test_files]
})

train_df.to_csv(DATA_DIR / 'train_set.csv', index=False)
test_df.to_csv(DATA_DIR / 'test_set.csv', index=False)

print(f'\nTrain: {len(train_df)}   Test: {len(test_df)}')
for cls in CLASS_LABELS:
    c = sum(1 for _, r in train_df.iterrows() if r['label'] == cls)
    if c > 0:
        print(f'  {cls:25s} {c:5d}')

## Cell 5 — Build Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Dense, BatchNormalization, Dropout,
    GlobalAveragePooling2D, GlobalMaxPooling2D, Concatenate, Input
)
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1_l2

def build_model(input_shape=(224, 224, 3), n_classes=8, dropout_rate=0.5):
    conv_base = DenseNet201(
        weights='imagenet', include_top=False,
        input_shape=input_shape, pooling=None
    )
    conv_base.trainable = False

    inputs = Input(shape=input_shape)
    x = conv_base(inputs)
    x = Concatenate()([GlobalAveragePooling2D()(x), GlobalMaxPooling2D()(x)])
    x = BatchNormalization()(x)
    x = Dense(1024, activation='relu', kernel_regularizer=l1_l2(0.001))(x)
    x = Dropout(dropout_rate)(x)
    x = BatchNormalization()(x)
    x = Dense(512, activation='relu', kernel_regularizer=l1_l2(0.001))(x)
    x = Dropout(dropout_rate)(x)
    x = BatchNormalization()(x)
    outputs = Dense(n_classes, activation='softmax')(x)

    return Model(inputs=inputs, outputs=outputs), conv_base

model, conv_base = build_model()
model.summary()

## Cell 6 — Setup Generators + Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_df = pd.read_csv(DATA_DIR / 'train_set.csv')
test_df = pd.read_csv(DATA_DIR / 'test_set.csv')

train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.12,
    brightness_range=(0.85, 1.15),
)
test_gen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    train_df,
    directory=str(IMAGES_DIR),
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True,
    seed=42,
)
test_flow = test_gen.flow_from_dataframe(
    test_df,
    directory=str(IMAGES_DIR),
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False,
)

class_weights = compute_class_weight(
    'balanced',
    classes=np.array(list(train_flow.class_indices.keys())),
    y=train_df['label'].values
)
cw_dict = {i: float(class_weights[i]) for i in range(len(class_weights))}

label_names = dict(train_flow.class_indices)
print('Class weights:')
for k, v in label_names.items():
    print(f'  {k:25s} {cw_dict[v]:.3f}')

## Cell 7 — Phase 1 Training (head only, 25 epochs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

h1 = model.fit(
    train_flow,
    validation_data=test_flow,
    epochs=25,
    class_weight=cw_dict,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            'p1_best.weights.h5',
            monitor='val_accuracy',
            save_best_only=True,
            save_weights_only=True,
            mode='max',
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
    ],
    verbose=1,
)

best_phase1 = max(h1.history['val_accuracy'])
print(f'\nBest Phase 1 val accuracy: {best_phase1:.4f}')

## Cell 8 — Phase 2 Training (fine-tune, 60 epochs)

In [ ]:
for layer in conv_base.layers[-50:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

h2 = model.fit(
    train_flow,
    validation_data=test_flow,
    epochs=60,
    initial_epoch=25,
    class_weight=cw_dict,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            'p2_best.weights.h5',
            monitor='val_accuracy',
            save_best_only=True,
            save_weights_only=True,
            mode='max',
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=12,
            restore_best_weights=True,
            verbose=1,
        ),
    ],
    verbose=1,
)

best_phase2 = max(h2.history['val_accuracy'])
print(f'\nBest Phase 2 val accuracy: {best_phase2:.4f}')

## Cell 9 — Save + Download

In [ ]:
import time
ts = time.strftime('%Y%m%d_%H%M%S')

model.save(f'breast_model_{ts}.h5')
model.save_weights(f'breast_weights_{ts}.h5')

!cp breast_model_{ts}.h5 "$WORK_DIR/"
!cp breast_weights_{ts}.h5 "$WORK_DIR/"

print(f'Saved:')
print(f'  breast_model_{ts}.h5')
print(f'  breast_weights_{ts}.h5')
print(f'\nAlso in Google Drive: {WORK_DIR}/')

from google.colab import files
files.download(f'breast_weights_{ts}.h5')

## Cell 10 — Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

test_flow.reset()
y_prob = model.predict(test_flow, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_flow.classes
names = list(test_flow.class_indices.keys())

acc = np.mean(y_pred == y_true)
print(f'\nAccuracy: {acc:.4f}  ({acc*100:.2f}%)')
print(f'\nClassification Report:\n')
print(classification_report(y_true, y_pred, target_names=names, zero_division=0))

# Training curves
full_history = {}
for k in h1.history:
    full_history[k] = h1.history[k] + h2.history[k] if k in h2.history else h1.history[k]

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(full_history['loss'], label='Train')
plt.plot(full_history['val_loss'], label='Val')
plt.axvline(x=25, color='gray', linestyle='--', alpha=0.5, label='Phase 2 start')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(full_history['accuracy'], label='Train')
plt.plot(full_history['val_accuracy'], label='Val')
plt.axvline(x=25, color='gray', linestyle='--', alpha=0.5, label='Phase 2 start')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
files.download('training_curves.png')

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
short = [n.replace('Density', 'D') for n in names]
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short, yticklabels=short)
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
files.download('confusion_matrix.png')

print('\nDone! All files saved.')